# Archived representative sampling v3\n\nHistorical configuration used while developing representative slice selection. This public notebook retains the thesis research implementation, but the clinical dataset, derived volumes, annotations, labels, identifiers, checkpoints, and executed outputs are not included.\n\n## Configuration\nSet the paths below only to data for which you have appropriate authorisation. Do not commit local paths or generated clinical outputs.\n

In [ ]:
from pathlib import Path\n\nDATA_ROOT = Path(\"/path/to/authorised/data\")\nOUTPUT_ROOT = Path(\"./outputs\")\nOUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\n

In [ ]:
# Google Colab Drive import removed for the public copy.\n# Configure DATA_ROOT below instead of mounting a private drive.


In [ ]:
import os, re
from pathlib import Path
import pandas as pd
import numpy as np


EXPECTED_PATIENTS = 37
DR = str(DATA_ROOT)

MUCUS_ROOT  = f"{DR}/MUCUS"
MUCUS2_ROOT = f"{DR}/MUCUS_2"
OUT_ROOT    = f"{DR}/mucus_master_37_fast"
Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)

MASTER_INDEX_PATH = f"{OUT_ROOT}/master_patient_index.csv"
LABELS_CSV_PATH   = f"{OUT_ROOT}/labels.csv"


# labels from excel
ANNOT_XLSX = f"{MUCUS_ROOT}/Annotations(2).xlsx"
lab = pd.read_excel(ANNOT_XLSX)

def norm_col(s): return re.sub(r"\s+", " ", str(s)).strip().lower()
lab = lab.rename(columns={c: norm_col(c) for c in lab.columns})

pid_col = "paziente" if "paziente" in lab.columns else None
if pid_col is None:
    raise RuntimeError(f"Can't find patient column in {ANNOT_XLSX}. Columns: {list(lab.columns)}")

if "media" in lab.columns:
    label_col = "media"
else:
    op_cols = [c for c in lab.columns if "operatore" in c or "operator" in c]
    lab["media"] = lab[op_cols].astype(float).mean(axis=1)
    label_col = "media"

labels = lab[[pid_col, label_col]].copy()
labels.columns = ["patient_id", "label"]
labels["patient_id"] = labels["patient_id"].astype(str).str.extract(r"(\d+)", expand=False)
labels = labels.dropna(subset=["patient_id"]).copy()
labels["patient_id"] = labels["patient_id"].astype(int).astype(str).str.zfill(4)
labels["label"] = labels["label"].astype(float)
labels = labels.drop_duplicates("patient_id", keep="first").reset_index(drop=True)
labels.to_csv(LABELS_CSV_PATH, index=False)


# patient folders
def extract_patient_id(name: str) -> str:
    m = re.search(r"(\d+)", name)
    return str(m.group(1)).zfill(4) if m else ""

rows = []
for root in [MUCUS_ROOT, MUCUS2_ROOT]:
    for d in Path(root).iterdir():
        if d.is_dir():
            pid = extract_patient_id(d.name)
            if pid:
                rows.append({"patient_id": pid, "patient_dir": str(d), "source_root": root})

df_pat = pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)

dups = df_pat[df_pat.duplicated("patient_id", keep=False)]
if len(dups):
    raise RuntimeError("Duplicate patient_id found across roots:\n" +
                       dups[["patient_id","patient_dir"]].to_string(index=False))

if len(df_pat) != EXPECTED_PATIENTS:
    print(f"WARNING: Expected {EXPECTED_PATIENTS}, found {len(df_pat)}")

df_master = df_pat.merge(labels, on="patient_id", how="left")

missing = df_master[df_master["label"].isna()][["patient_id","patient_dir"]]
if len(missing):
    raise RuntimeError("Missing labels for some patients:\n" + missing.to_string(index=False))

df_master.to_csv(MASTER_INDEX_PATH, index=False)

print("Wrote:")
print(" -", MASTER_INDEX_PATH)
print(" -", LABELS_CSV_PATH)


In [ ]:

OUT_ROOT = str(DATA_ROOT)
df = pd.read_csv(f"{OUT_ROOT}/master_patient_index.csv")
df["patient_id"] = df["patient_id"].astype(str).str.zfill(4)


# pydicom
!pip -q install pydicom
import pydicom

SKIP_DIR_NAMES = {"__macosx", ".ipynb_checkpoints", "cache", "logs", "tmp", "temp"}
SKIP_FILE_EXT  = {".txt",".csv",".json",".xml",".pdf",".zip",".rar",".7z",".html",
                  ".doc",".docx",".ppt",".pptx",".xls",".xlsx"}

BAD_SERIES = ["scout","localizer","topogram","surview"]


def iter_files(root: Path):
    for p in root.rglob("*"):
        if p.is_dir():
            if p.name.lower() in SKIP_DIR_NAMES:
                continue
            continue
        if p.name.startswith("."):
            continue
        if p.suffix.lower() in SKIP_FILE_EXT:
            continue
        yield p


def try_read_dicom_header(fp: Path):
    try:
        ds = pydicom.dcmread(str(fp), stop_before_pixels=True, force=True)
        if getattr(ds, "SOPInstanceUID", None) is None and getattr(ds, "SeriesInstanceUID", None) is None:
            return None
        return ds
    except Exception:
        return None


def sample_valid_dicoms(patient_dir: str, max_dicoms=25, max_checks=8000):
    out = []
    for i, fp in enumerate(iter_files(Path(patient_dir))):
        if i >= max_checks:
            break
        ds = try_read_dicom_header(fp)
        if ds is not None:
            out.append(ds)
            if len(out) >= max_dicoms:
                break
    return out


def summarize_header(ds):
    def g(attr, default=None): return getattr(ds, attr, default)
    series_desc = g("SeriesDescription")
    series_desc_l = str(series_desc).lower() if series_desc is not None else ""
    return {
        "modality": g("Modality"),
        "manufacturer": g("Manufacturer"),
        "model": g("ManufacturerModelName"),
        "kernel": g("ConvolutionKernel"),
        "slice_thickness": g("SliceThickness"),
        "pixel_spacing": g("PixelSpacing"),
        "spacing_between": g("SpacingBetweenSlices"),
        "kvp": g("KVP"),
        "series_desc": series_desc,
        "looks_like_scout": any(k in series_desc_l for k in BAD_SERIES),
    }

rows = []
for _, r in df.iterrows():
    pid = r["patient_id"]
    pdir = r["patient_dir"]
    root = Path(pdir)

    exts = []
    total_files = 0
    for fp in iter_files(root):
        total_files += 1
        exts.append(fp.suffix.lower())

    if total_files == 0:
        rows.append({"patient_id": pid, "patient_dir": pdir, "status": "EXCLUDE", "reason": "EMPTY_FOLDER",
                     "total_files": 0, "jpg": 0, "png": 0, "dcm_ext": 0, "other_ext": 0})
        continue

    ext_ser = pd.Series(exts)
    n_jpg = int((ext_ser == ".jpg").sum() + (ext_ser == ".jpeg").sum())
    n_png = int((ext_ser == ".png").sum())
    n_dcm_ext = int((ext_ser == ".dcm").sum())
    n_other = int(total_files - n_jpg - n_png - n_dcm_ext)

    samp = sample_valid_dicoms(pdir, max_dicoms=25)
    if len(samp) == 0:
        reason = "NON_DICOM_JPG" if n_jpg > 0 and (n_dcm_ext == 0) else "NO_DICOM_OTHER"
        rows.append({"patient_id": pid, "patient_dir": pdir, "status": "EXCLUDE", "reason": reason,
                     "total_files": total_files, "jpg": n_jpg, "png": n_png, "dcm_ext": n_dcm_ext, "other_ext": n_other})
        continue

    hdf = pd.DataFrame([summarize_header(ds) for ds in samp])

    def pick(col):
        s = hdf[col].dropna()
        return s.iloc[0] if len(s) else np.nan

    rows.append({
        "patient_id": pid,
        "patient_dir": pdir,
        "status": "OK",
        "reason": "OK_DICOM_FOUND",
        "total_files": total_files,
        "jpg": n_jpg,
        "png": n_png,
        "dcm_ext": n_dcm_ext,
        "other_ext": n_other,
        "sampled_dicoms": len(samp),
        "modality": pick("modality"),
        "manufacturer": pick("manufacturer"),
        "model": pick("model"),
        "kernel": pick("kernel"),
        "slice_thickness": pick("slice_thickness"),
        "pixel_spacing": pick("pixel_spacing"),
        "spacing_between": pick("spacing_between"),
        "kvp": pick("kvp"),
        "example_series_desc": pick("series_desc"),
        "any_sample_looks_like_scout": bool(hdf["looks_like_scout"].fillna(False).any()),
    })

card = pd.DataFrame(rows)


# Hard overrides (based on manual verification)
card.loc[card["patient_id"] == \"REDACTED_PATIENT_ID\", ["status","reason"]] = ["EXCLUDE","EMPTY_FOLDER"]
card.loc[card["patient_id"] == \"REDACTED_PATIENT_ID\", ["status","reason"]] = ["EXCLUDE","NON_DICOM_JPG"]

card_path = f"{OUT_ROOT}/data_card_v2.csv"
card.to_csv(card_path, index=False)

excluded = card[card["status"] != "OK"][["patient_id","reason","total_files","jpg","png","dcm_ext","other_ext"]]
excluded_path = f"{OUT_ROOT}/excluded_patients.csv"
excluded.to_csv(excluded_path, index=False)


print("Wrote:")
print(" -", card_path)
print(" -", excluded_path)
print("\nCounts:\n", card["status"].value_counts().to_string())
print("\nExcluded reasons:\n", excluded["reason"].value_counts().to_string())



In [ ]:
# --- Label distribution + diagnostics ---

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


OUT_ROOT = str(DATA_ROOT)

df_master = pd.read_csv(f"{OUT_ROOT}/master_patient_index.csv")
df_master["patient_id"] = df_master["patient_id"].astype(str).str.zfill(4)
df_master["label"] = df_master["label"].astype(float)

card = pd.read_csv(f"{OUT_ROOT}/data_card_v2.csv")
card["patient_id"] = card["patient_id"].astype(str).str.zfill(4)


# CT-usable cohort (status OK)
ok_ids = set(card.loc[card["status"] == "OK", "patient_id"].tolist())

df_ok = df_master[df_master["patient_id"].isin(ok_ids)].copy()

def summarize_and_plot(df, title_suffix):
    y = df["label"].to_numpy()

    print(f"\n--- {title_suffix} ---")
    print("n patients:", len(df))
    print("min/max:", float(np.min(y)), float(np.max(y)))
    print("mean/std:", float(np.mean(y)), float(np.std(y, ddof=1)))
    print("median:", float(np.median(y)))
    print("quantiles 10/25/75/90:", [float(np.quantile(y, q)) for q in [0.1, 0.25, 0.75, 0.9]])

    freq = df["label"].value_counts().sort_index()
    print("\nLabel frequency:")
    print(freq.to_string())

    plt.figure()
    if freq.shape[0] <= 20:
        plt.bar(freq.index.astype(str), freq.values)
        plt.xticks(rotation=45, ha="right")
    else:
        plt.hist(y, bins=10)

    plt.xlabel("Label (Median)")
    plt.ylabel("Count")
    plt.title(f"Label distribution — {title_suffix}")
    plt.tight_layout()
    plt.show()



# Label distribution for CT-usable cohort (training cohort)
summarize_and_plot(df_ok, "CT-usable patients only")

# Print excluded patients + their labels (important documentation)
excluded = card[card["status"] != "OK"][["patient_id","reason"]].merge(
    df_master[["patient_id","label"]], on="patient_id", how="left"
).sort_values("patient_id")

print("\nExcluded patients (and their labels):")
display(excluded)


In [ ]:

OUT_ROOT = str(DATA_ROOT)

# Load master + data_card_v2 and keep only usable patients
df = pd.read_csv(f"{OUT_ROOT}/master_patient_index.csv")
df["patient_id"] = df["patient_id"].astype(str).str.zfill(4)

card = pd.read_csv(f"{OUT_ROOT}/data_card_v2.csv")
card["patient_id"] = card["patient_id"].astype(str).str.zfill(4)

ok_ids = set(card.loc[card["status"] == "OK", "patient_id"].tolist())
df = df[df["patient_id"].isin(ok_ids)].copy()

SKIP_DIR_NAMES = {"__macosx", ".ipynb_checkpoints", "cache", "logs", "tmp", "temp"}

def estimate_max_slices(patient_dir: str, min_files_in_dir: int = 30):
    root = Path(patient_dir)
    best = 0
    for d in root.rglob("*"):
        if not d.is_dir():
            continue
        if d.name.lower() in SKIP_DIR_NAMES:
            continue
        try:
            n = sum(1 for p in d.iterdir() if p.is_file() and not p.name.startswith("."))
        except Exception:
            continue
        if n >= min_files_in_dir:
            best = max(best, n)
    if best == 0:
        try:
            best = sum(1 for p in root.iterdir() if p.is_file() and not p.name.startswith("."))
        except Exception:
            best = 0
    return best

df["max_slices_est"] = df["patient_dir"].apply(estimate_max_slices)

table = (
    df[["patient_id", "max_slices_est"]]
    .sort_values(["max_slices_est", "patient_id"], ascending=[False, True])
    .reset_index(drop=True)
)

# Pretty describe()
desc = table["max_slices_est"].describe()
desc.loc[["count","min","25%","50%","75%","max"]] = desc.loc[["count","min","25%","50%","75%","max"]].astype(int)
desc.loc[["mean","std"]] = desc.loc[["mean","std"]].round(2)
print(desc.to_string())

display(table)

# Save
out_path = f"{OUT_ROOT}/max_slices_est_ct_usable.csv"
table.to_csv(out_path, index=False)
print("Saved:", out_path)




In [ ]:
# 00 - Setup paths + load usable cohort before preprocessing

import os
import pandas as pd
from pathlib import Path

OUT_ROOT = str(DATA_ROOT)
MASTER_PATH = f"{OUT_ROOT}/master_patient_index.csv"
CARD_V2_PATH = f"{OUT_ROOT}/data_card_v2.csv"

assert os.path.exists(MASTER_PATH), "Missing master_patient_index.csv"
assert os.path.exists(CARD_V2_PATH), "Missing data_card_v2.csv"

master = pd.read_csv(MASTER_PATH)
master["patient_id"] = master["patient_id"].astype(str).str.zfill(4)

card = pd.read_csv(CARD_V2_PATH)
card["patient_id"] = card["patient_id"].astype(str).str.zfill(4)

usable_ids = set(card.loc[card["status"]=="OK", "patient_id"])
df = master[master["patient_id"].isin(usable_ids)].copy().reset_index(drop=True)

print("Usable patients:", len(df))
print("Example rows:")
display(df.head(10))



In [ ]:
# 01 - Lock folds (patient-level) and save split file

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, KFold

SPLIT_PATH = f"{OUT_ROOT}/splits_5fold_locked.csv"

y = df["label"].astype(float).values
seed = 42
n_splits = 5

def try_bins(y, max_bins=5):
    # Try qcut bins, reduce bins until each bin has at least 2 samples
    for nb in range(max_bins, 1, -1):
        try:
            b = pd.qcut(y, q=nb, duplicates="drop").codes
            if np.min(np.bincount(b)) >= 2:
                return b, f"qcut_{nb}"
        except Exception:
            pass
    return None, "no_bins"

bins, bin_method = try_bins(y, max_bins=5)

fold = np.full(len(df), -1, dtype=int)

if bins is not None:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for k, (_, test_idx) in enumerate(skf.split(np.zeros(len(df)), bins)):
        fold[test_idx] = k
    method = f"StratifiedKFold({bin_method})"
else:
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for k, (_, test_idx) in enumerate(kf.split(np.zeros(len(df)))):
        fold[test_idx] = k
    method = "KFold_fallback"

splits = df[["patient_id","label"]].copy()
splits["fold"] = fold
splits["seed"] = seed
splits["method"] = method
splits.to_csv(SPLIT_PATH, index=False)

print("Wrote:", SPLIT_PATH)
print("Method:", method)
print(splits["fold"].value_counts().sort_index().to_dict())


In [ ]:
# 03 - Build series_index.csv (all series per patient)

import numpy as np
import pandas as pd
from pathlib import Path
import pydicom

SERIES_INDEX_PATH = f"{OUT_ROOT}/series_index.csv"

SKIP_DIR_NAMES = {"__macosx", ".ipynb_checkpoints", "cache", "logs", "tmp", "temp"}
SKIP_FILE_EXT  = {".txt",".csv",".json",".xml",".pdf",".zip",".rar",".7z",".html",
                  ".doc",".docx",".ppt",".pptx",".xls",".xlsx"}

def iter_files(root: Path):
    for p in root.rglob("*"):
        if p.is_dir():
            if p.name.lower() in SKIP_DIR_NAMES:
                continue
            continue
        if p.name.startswith("."):
            continue
        if p.suffix.lower() in SKIP_FILE_EXT:
            continue
        yield p

def _as_float_list(x, n=None):
    try:
        if x is None:
            return None
        if isinstance(x, (list, tuple)):
            arr = [float(v) for v in x]
        else:
            s = str(x).replace("[","").replace("]","").replace("(","").replace(")","")
            arr = [float(v) for v in s.replace("\\", ",").split(",") if len(v.strip()) > 0]
        if n is not None and len(arr) != n:
            return None
        return arr
    except Exception:
        return None

def _axial_score_from_iop(iop6):
    if iop6 is None:
        return np.nan
    try:
        r = np.array(iop6[:3], dtype=float)
        c = np.array(iop6[3:], dtype=float)
        n = np.cross(r, c)
        n = n / (np.linalg.norm(n) + 1e-8)
        return float(abs(n[2]))  # 1.0 best axial
    except Exception:
        return np.nan

def read_hdr(fp: Path):
    try:
        ds = pydicom.dcmread(str(fp), stop_before_pixels=True, force=True)
        if getattr(ds, "SeriesInstanceUID", None) is None:
            return None

        def g(a, d=None): return getattr(ds, a, d)

        iop = _as_float_list(g("ImageOrientationPatient", None), n=6)
        axial_score = _axial_score_from_iop(iop)

        # selection-friendly tags
        body_part = str(g("BodyPartExamined", "") or "")
        protocol  = str(g("ProtocolName", "") or "")
        study_desc = str(g("StudyDescription", "") or "")

        # image type
        image_type = g("ImageType", "")
        if isinstance(image_type, (list, tuple)):
            image_type = "\\".join([str(v) for v in image_type])
        image_type = str(image_type)

        return {
            "path": str(fp),
            "Modality": g("Modality"),
            "SeriesInstanceUID": g("SeriesInstanceUID"),
            "StudyInstanceUID": g("StudyInstanceUID"),
            "SeriesDescription": g("SeriesDescription"),
            "StudyDescription": study_desc,
            "ProtocolName": protocol,
            "BodyPartExamined": body_part,
            "ImageType": image_type,
            "InstanceNumber": g("InstanceNumber"),
            "SliceThickness": g("SliceThickness"),
            "PixelSpacing": g("PixelSpacing"),
            "SpacingBetweenSlices": g("SpacingBetweenSlices"),
            "ConvolutionKernel": g("ConvolutionKernel"),
            "KVP": g("KVP"),
            "Rows": g("Rows"),
            "Columns": g("Columns"),
            "axial_score": axial_score,
        }
    except Exception:
        return None

rows = []
for _, r in df.iterrows():
    pid = str(r["patient_id"]).zfill(4)
    pdir = Path(r["patient_dir"])
    n_checked = 0

    for fp in iter_files(pdir):
        n_checked += 1
        h = read_hdr(fp)
        if h is not None:
            rows.append({"patient_id": pid, **h})
        if n_checked >= 100000:
            break

raw = pd.DataFrame(rows)
if raw.empty:
    raise RuntimeError("No DICOM headers found. Check paths and permissions.")

def first_nonnull(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else np.nan

series = (raw.groupby(["patient_id","SeriesInstanceUID"], as_index=False)
            .agg(
                n_files=("path","count"),
                modality=("Modality", first_nonnull),
                study_uid=("StudyInstanceUID", first_nonnull),
                series_desc=("SeriesDescription", first_nonnull),
                study_desc=("StudyDescription", first_nonnull),
                protocol=("ProtocolName", first_nonnull),
                body_part=("BodyPartExamined", first_nonnull),
                image_type=("ImageType", first_nonnull),
                rows=("Rows", first_nonnull),
                cols=("Columns", first_nonnull),
                slice_thickness=("SliceThickness", "median"),
                pixel_spacing=("PixelSpacing", first_nonnull),
                spacing_between=("SpacingBetweenSlices", first_nonnull),
                kernel=("ConvolutionKernel", first_nonnull),
                kvp=("KVP", first_nonnull),
                axial_score=("axial_score", "median"),
            ))

series.to_csv(SERIES_INDEX_PATH, index=False)
print("Wrote:", SERIES_INDEX_PATH)
print("Patients with at least 1 series:", series["patient_id"].nunique(), "/", df["patient_id"].nunique())
display(series.sort_values(["patient_id","n_files"], ascending=[True, False]).head(5))



In [ ]:
# 04 - Select the “best” CT axial chest/lung series per patient → selected_series.csv [UPDATED]

import numpy as np
import pandas as pd

SELECTED_SERIES_PATH = f"{OUT_ROOT}/selected_series.csv"
series = pd.read_csv(SERIES_INDEX_PATH)
series["patient_id"] = series["patient_id"].astype(str).str.zfill(4)

# Exclusions
BAD = ["scout", "localizer", "topogram", "surview", "dose", "report"]
BAD_ORIENT = ["cor", "coronal", "sag", "sagittal", "mpr", "reformat", "reformatted", "3d", "mip", "minip"]
BAD_ANATOMY = ["head", "brain", "skull", "sinus", "neck", "cspine", "tspine", "lspine", "abdomen", "pelvis", "liver"]

GOOD = ["lung", "chest", "thorax", "thoracic", "pulmo", "axial"]

def _txt(*xs):
    return " ".join([str(x or "").lower() for x in xs])

def score_row(row):
    mod  = str(row.get("modality","")).upper()
    desc = str(row.get("series_desc","") or "")
    ityp = str(row.get("image_type","") or "")
    body = str(row.get("body_part","") or "")
    prot = str(row.get("protocol","") or "")
    styd = str(row.get("study_desc","") or "")

    t = _txt(desc, ityp, body, prot, styd)

    score = float(row.get("n_files", 0))

    # Must be CT
    if mod != "CT":
        return -1e12

    # Hard negatives
    if any(k in t for k in BAD):
        score -= 1e6
    if "localizer" in ityp.lower() or "LOCALIZER" in ityp:
        score -= 2e6  # extra hard exclude if ImageType says so
    if "derived" in ityp.lower() or "secondary" in ityp.lower():
        score -= 2e5
    if any(k in t for k in BAD_ORIENT):
        score -= 5e4
    if any(k in t for k in BAD_ANATOMY):
        score -= 2e5

    # Prefer chest/lung keywords
    if any(k in t for k in GOOD):
        score += 2000

    # Prefer axial by IOP-derived score
    axial_score = row.get("axial_score", np.nan)
    if not np.isnan(axial_score):
        score += 5e4 * float(axial_score)

    # Slice thickness preference (airway tasks generally prefer <= 2-3mm)
    try:
        th = float(row.get("slice_thickness"))
        if 0.3 <= th <= 3.0:
            score += 1200
        elif th <= 4.0:
            score += 200
        elif th > 6.0:
            score -= 1500
        else:
            score -= 600
    except Exception:
        pass

    # Typical CT size preference (512x512)
    try:
        r = int(row.get("rows", 0) or 0)
        c = int(row.get("cols", 0) or 0)
        if (r, c) == (512, 512):
            score += 300
    except Exception:
        pass

    return score

ct = series[series["modality"].astype(str).str.upper() == "CT"].copy()
ct["score"] = ct.apply(score_row, axis=1)

best = (ct.sort_values(["patient_id","score"], ascending=[True, False])
          .groupby("patient_id", as_index=False)
          .head(1)
          .reset_index(drop=True))

best.to_csv(SELECTED_SERIES_PATH, index=False)
print("Wrote:", SELECTED_SERIES_PATH)

cols = ["patient_id","SeriesInstanceUID","n_files","series_desc","study_desc","protocol","body_part",
        "slice_thickness","pixel_spacing","axial_score","kernel","score"]
display(best[cols].head(5))



In [ ]:
import torch

!pip -q install pydicom SimpleITK
import pydicom
import SimpleITK as sitk
print("pydicom:", pydicom.__version__)
print("SimpleITK:", sitk.Version_VersionString())



!pip -q install gdcm pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg

In [ ]:
# 05A - Rebuild volumes (HU) using selected_series first + robust ordering + per-slice rescale

import numpy as np
import pandas as pd
from pathlib import Path
import pydicom
import SimpleITK as sitk

OUT_ROOT = str(DATA_ROOT)
VOL_DIR = f"{OUT_ROOT}/volumes_hu"
Path(VOL_DIR).mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = f"{OUT_ROOT}/volume_manifest_.csv"
SELECTED_SERIES_PATH = f"{OUT_ROOT}/selected_series.csv"

df = pd.read_csv(f"{OUT_ROOT}/master_patient_index.csv")
df["patient_id"] = df["patient_id"].astype(str).str.zfill(4)

card = pd.read_csv(f"{OUT_ROOT}/data_card_v2.csv")
card["patient_id"] = card["patient_id"].astype(str).str.zfill(4)
usable_ids = set(card.loc[card["status"] == "OK", "patient_id"])
df = df[df["patient_id"].isin(usable_ids)].reset_index(drop=True)

selected = pd.read_csv(SELECTED_SERIES_PATH)
selected["patient_id"] = selected["patient_id"].astype(str).str.zfill(4)
sel_map = dict(zip(selected["patient_id"], selected["SeriesInstanceUID"]))

SKIP_DIR_NAMES = {"__macosx", ".ipynb_checkpoints", "cache", "logs", "tmp", "temp"}
SKIP_FILE_EXT  = {".txt",".csv",".json",".xml",".pdf",".zip",".rar",".7z",".html",
                  ".doc",".docx",".ppt",".pptx",".xls",".xlsx"}

def iter_files(root: Path):
    for p in root.rglob("*"):
        if p.is_dir():
            if p.name.lower() in SKIP_DIR_NAMES:
                continue
            continue
        if p.name.startswith("."):
            continue
        if p.suffix.lower() in SKIP_FILE_EXT:
            continue
        yield p


def _as_float_list(x, n=None):
    try:
        if x is None:
            return None
        if isinstance(x, (list, tuple)):
            arr = [float(v) for v in x]
        else:
            s = str(x).replace("[","").replace("]","").replace("(","").replace(")","")
            arr = [float(v) for v in s.replace("\\", ",").split(",") if len(v.strip()) > 0]
        if n is not None and len(arr) != n:
            return None
        return arr
    except Exception:
        return None


def _normal_from_iop(iop6):
    if iop6 is None:
        return None
    r = np.array(iop6[:3], dtype=float)
    c = np.array(iop6[3:], dtype=float)
    n = np.cross(r, c)
    n = n / (np.linalg.norm(n) + 1e-8)
    return n


def read_uid_quick(fp: Path):
    try:
        ds = pydicom.dcmread(str(fp), stop_before_pixels=True, force=True)
        uid = getattr(ds, "SeriesInstanceUID", None)
        if uid is None:
            return None
        return uid
    except Exception:
        return None


def read_sort_meta(fp: str):
    """
    Header-only meta needed for robust sorting + per-slice rescale.
    """
    ds = pydicom.dcmread(fp, stop_before_pixels=True, force=True)
    inst = int(getattr(ds, "InstanceNumber", 0) or 0)

    iop = _as_float_list(getattr(ds, "ImageOrientationPatient", None), n=6)
    ipp = _as_float_list(getattr(ds, "ImagePositionPatient", None), n=3)

    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)

    return inst, iop, ipp, slope, intercept


def build_hu_volume(files_sorted):
    """
    Read with SimpleITK (handles compression). Then:
    - Detect if values already look HU-like (contain large negatives).
    - If not HU-like, apply per-slice slope/intercept from DICOM headers.
    Returns: int16 HU-ish volume and spacing (x,y,z).
    """
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(files_sorted)
    reader.MetaDataDictionaryArrayUpdateOn()
    reader.LoadPrivateTagsOn()
    img = reader.Execute()
    arr = sitk.GetArrayFromImage(img).astype(np.float32)  # (Z,H,W)

    sx, sy, sz = img.GetSpacing()

    # Heuristic: if we already see HU-like negatives, do NOT apply rescale again
    # (raw CT stored pixels are usually non-negative; HU often includes negatives ~ -1000)
    already_hu_like = (np.nanpercentile(arr, 1) < -200)

    slopes = []
    intercepts = []
    for f in files_sorted:
        try:
            ds = pydicom.dcmread(f, stop_before_pixels=True, force=True)
            slopes.append(float(getattr(ds, "RescaleSlope", 1.0) or 1.0))
            intercepts.append(float(getattr(ds, "RescaleIntercept", 0.0) or 0.0))
        except Exception:
            slopes.append(1.0)
            intercepts.append(0.0)

    slopes = np.asarray(slopes, dtype=np.float32)
    intercepts = np.asarray(intercepts, dtype=np.float32)

    if not already_hu_like:
        arr = arr * slopes[:, None, None] + intercepts[:, None, None]

    return arr.astype(np.int16), (float(sx), float(sy), float(sz)), bool(already_hu_like)


manifest_rows = []

for _, row in df.iterrows():
    pid = row["patient_id"]
    pdir = Path(row["patient_dir"])

    # group file paths by SeriesInstanceUID
    uid_files = {}
    for fp in iter_files(pdir):
        uid = read_uid_quick(fp)
        if uid is None:
            continue
        uid_files.setdefault(uid, []).append(str(fp))

    if not uid_files:
        manifest_rows.append({"patient_id": pid, "status": "FAIL_NO_SERIES", "volume_path": ""})
        continue

    # candidate order: selected uid first (if present), then others by file count
    candidates = []
    sel_uid = sel_map.get(pid, None)
    if sel_uid is not None and sel_uid in uid_files:
        candidates.append(sel_uid)

    others = [u for u in uid_files.keys() if u != sel_uid]
    others = sorted(others, key=lambda u: len(uid_files[u]), reverse=True)
    candidates.extend(others[:4])  # up to 5 total

    success = False
    last_err = ""

    for uid in candidates:
        files = uid_files[uid]
        if len(files) < 10:
            last_err = f"TOO_FEW_FILES({len(files)})"
            continue

        # read meta for sorting (header-only)
        metas = []
        for f in files:
            try:
                inst, iop, ipp, slope, intercept = read_sort_meta(f)
                metas.append((f, inst, iop, ipp))
            except Exception:
                continue

        if len(metas) < 10:
            last_err = "META_READ_FAILED"
            continue

        # use first valid IOP to define slice normal
        iop0 = None
        for _, _, iop, _ in metas:
            if iop is not None:
                iop0 = iop
                break
        n = _normal_from_iop(iop0)

        # robust position scalar for sorting
        sort_keys = []
        for f, inst, iop, ipp in metas:
            if n is not None and ipp is not None:
                pos = float(np.dot(np.array(ipp, dtype=float), n))
            else:
                pos = float(inst)
            sort_keys.append((pos, inst, f))

        sort_keys.sort(key=lambda x: (x[0], x[1], x[2]))
        files_sorted = [f for _, _, f in sort_keys]

        try:
            vol_hu, (sx, sy, sz), already_hu_like = build_hu_volume(files_sorted)

            if vol_hu.shape[0] < 10:
                last_err = f"TOO_FEW_SLICES({vol_hu.shape[0]})"
                continue

            out_path = f"{VOL_DIR}/{pid}.npy"
            np.save(out_path, vol_hu)

            manifest_rows.append({
                "patient_id": pid,
                "status": "OK",
                "volume_path": out_path,
                "n_files_in_series": int(len(files_sorted)),
                "n_slices": int(vol_hu.shape[0]),
                "shape": str(tuple(vol_hu.shape)),
                "spacing_x_mm": float(sx),
                "spacing_y_mm": float(sy),
                "spacing_z_mm": float(sz),
                "used_series_uid": uid,
                "used_selected_uid": sel_uid if sel_uid is not None else "",
                "already_hu_like": int(already_hu_like),
                "hu_min": int(np.min(vol_hu)),
                "hu_max": int(np.max(vol_hu)),
            })
            success = True
            break

        except Exception as e:
            last_err = str(e)[:200]
            continue

    if not success:
        manifest_rows.append({
            "patient_id": pid,
            "status": "FAIL_ALL_SERIES",
            "volume_path": "",
            "error": last_err
        })


manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(MANIFEST_PATH, index=False)

print("Wrote:", MANIFEST_PATH)
print("OK volumes:", int((manifest["status"]=="OK").sum()), "/", len(manifest))
display(manifest["status"].value_counts())
display(manifest.head(5))



In [ ]:
# 05B — Quick visual QA per patient (fixed windows: lung + mediastinum)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


OUT_ROOT = str(DATA_ROOT)
MANIFEST_PATH = f"{OUT_ROOT}/volume_manifest_.csv"


# Display windows (HU)
WINDOWS = {
    "lung": (-1350, 150),          # shows anatomy
    "mediastinum": (-160, 240),    # shows structures
}


def show_patient_slices(vol, pid, n_rand=3, seed=7):
    Z = vol.shape[0]
    rng = np.random.default_rng(seed)

    fixed = [0, Z//2, Z-1]
    fixed_set = set(fixed)

    # sample random slices excluding fixed ones
    candidates = np.array([z for z in range(Z) if z not in fixed_set], dtype=int)
    if len(candidates) > 0:
        rand = rng.choice(candidates, size=min(n_rand, len(candidates)), replace=False).tolist()
    else:
        rand = []
    idxs = fixed + rand

    n = len(idxs)
    nrows = len(WINDOWS)
    fig, axes = plt.subplots(nrows, n, figsize=(3.0*n, 3.0*nrows))

    if nrows == 1:
        axes = np.array([axes])
    if n == 1:
        axes = axes.reshape(nrows, 1)

    for row_i, (wname, (vmin, vmax)) in enumerate(WINDOWS.items()):
        for col_i, z in enumerate(idxs):
            ax = axes[row_i, col_i]
            ax.imshow(vol[z], cmap="gray", vmin=vmin, vmax=vmax)
            ax.set_title(f"{pid} | {wname}\nz={z}/{Z-1}")
            ax.axis("off")

    plt.tight_layout()
    plt.show()

man = pd.read_csv(MANIFEST_PATH)
man["patient_id"] = man["patient_id"].astype(str).str.zfill(4)
man_ok = man.query("status == 'OK'").copy().sort_values("patient_id")

print("OK volumes:", len(man_ok))

# number of patients to review
LIMIT = None

for _, r in man_ok.iterrows():
    pid = r["patient_id"]
    vp  = r["volume_path"]
    vol = np.load(vp, mmap_mode="r")  # (Z,H,W)

    print(f"\nPatient {pid}  shape={vol.shape}  spacing_z_mm={r.get('spacing_z_mm', 'NA')}")
    show_patient_slices(vol, pid, n_rand=3, seed=int(pid) % 1000)

    if LIMIT is not None:
        LIMIT -= 1
        if LIMIT <= 0:
            break



In [ ]:
# 06 - Build one unified training index (manifest + labels + folds)

import pandas as pd
import numpy as np


OUT_ROOT = str(DATA_ROOT)
MANIFEST_PATH = f"{OUT_ROOT}/volume_manifest_.csv"
SPLITS_PATH   = f"{OUT_ROOT}/splits_5fold_locked.csv"
MASTER_PATH   = f"{OUT_ROOT}/master_patient_index.csv"

man = pd.read_csv(MANIFEST_PATH)
man["patient_id"] = man["patient_id"].astype(str).str.zfill(4)
man = man.query("status == 'OK'").copy()

spl = pd.read_csv(SPLITS_PATH)
spl["patient_id"] = spl["patient_id"].astype(str).str.zfill(4)

master = pd.read_csv(MASTER_PATH)
master["patient_id"] = master["patient_id"].astype(str).str.zfill(4)
master["label"] = master["label"].astype(float)


# critical uniqueness checks (avoid duplicated patients)
dup_ok = man["patient_id"].duplicated().sum()
assert dup_ok == 0, f"Manifest has duplicate OK patient rows: {dup_ok}. Fix before building train_index."

dup_spl = spl["patient_id"].duplicated().sum()
assert dup_spl == 0, f"Splits file has duplicate patient rows: {dup_spl}. Fix splits_5fold_locked.csv."

dup_mas = master["patient_id"].duplicated().sum()
assert dup_mas == 0, f"Master has duplicate patient rows: {dup_mas}. Fix master_patient_index.csv."

df_index = (man.merge(master[["patient_id","label"]], on="patient_id", how="left", validate="one_to_one")
              .merge(spl[["patient_id","fold"]],   on="patient_id", how="left", validate="one_to_one"))


assert df_index["label"].notna().all(), "Some labels missing after merge."
assert df_index["fold"].notna().all(), "Some folds missing after merge."
assert df_index["volume_path"].astype(str).str.len().gt(0).all(), "Missing volume_path."
assert df_index["patient_id"].nunique() == len(df_index), "train_index has duplicate patients."

df_index = df_index.sort_values("patient_id").reset_index(drop=True)


print("Training index rows:", len(df_index))
display(df_index[["patient_id","label","fold","n_slices","spacing_x_mm","spacing_y_mm","spacing_z_mm",
                  "used_series_uid","used_selected_uid"]].head(12))

INDEX_PATH = f"{OUT_ROOT}/train_index.csv"
df_index.to_csv(INDEX_PATH, index=False)
print("Saved:", INDEX_PATH)



In [ ]:
# Cell 07 — Preprocessing utilities (2-window, normalize, resize)

import numpy as np
import cv2

IMG_SIZE = 256

# Windows in HU
LUNG_MIN, LUNG_MAX = -1350, 150
MEDIA_MIN, MEDIA_MAX = -160, 240

# Canonical HU clamp to remove domain differences (e.g., min -3024 / -2048)
HU_CANON_MIN, HU_CANON_MAX = -1350, 600


def hu_canonicalize(x_hu: np.ndarray) -> np.ndarray:
    # clamp to plausible HU range so all scans share the same floor/ceiling
    return np.clip(x_hu, HU_CANON_MIN, HU_CANON_MAX).astype(np.float32)


def _window_norm(x_hu: np.ndarray, hu_min: float, hu_max: float) -> np.ndarray:
    x = np.clip(x_hu, hu_min, hu_max).astype(np.float32)
    x = (x - hu_min) / (hu_max - hu_min + 1e-6)  # [0,1]
    return x


def _resize01(x: np.ndarray, size=IMG_SIZE) -> np.ndarray:
    x = cv2.resize(x, (size, size), interpolation=cv2.INTER_AREA)
    return np.clip(x, 0.0, 1.0).astype(np.float32)


def preprocess_slice(slice_hu: np.ndarray) -> np.ndarray:
    slice_hu = hu_canonicalize(slice_hu)

    lung = _resize01(_window_norm(slice_hu, LUNG_MIN, LUNG_MAX))
    medi = _resize01(_window_norm(slice_hu, MEDIA_MIN, MEDIA_MAX))
    x = np.stack([lung, medi], axis=0)  # (2, H, W)
    return x



In [ ]:
# Cell 08 — Representative slice sampling (QC filter + diversity + informative extremes)
# UPDATED: adds HU canonicalization in QC + BODY_FRAC filter to avoid body-dominant slices (e.g., 2226)

import numpy as np

# keep consistent-ish, but can now safely increase
CENTER_FRACTION = 0.65       # 0.6 was OK, 0.7 often improves by avoiding extreme ends
N_SLICES_REP    = 128        # good starting point; later tune 96/128/160/192
FEAT_HW         = 64         # low-res for diversity features (NOT 256; 64 is plenty)
N_PCA           = 24         # compress features before clustering
RANDOM_STATE    = 42

# Tune ??
LUNG_HU_THR    = -350        # HU below this is likely lung/air (very rough proxy)
MIN_LUNG_FRAC  = 0.03        # discard slices with tiny lung area (junk ends)
MAX_AIR_FRAC   = 0.98        # discard slices that are almost all air (rare but happens)
HI_DENS_THR    = -200        # “denser-than-expected” inside lung mask proxy (crudely)
N_INFORMATIVE  = 24          # always include top-N informative slices
MAX_BODY_FRAC  = 0.55        # NEW: discard slices dominated by body/soft-tissue (reduces false positives)

# HU canonical range for QC/heuristics (stabilizes mixed intensity domains)
HU_CANON_MIN, HU_CANON_MAX = -1350, 600


def _central_candidates(n_slices: int, center_fraction: float):
    if n_slices <= 0:
        return np.array([], dtype=int)
    if center_fraction < 1.0:
        margin = int((1 - center_fraction) * n_slices / 2)
        lo = max(margin, 0)
        hi = max(n_slices - margin, lo + 1)
    else:
        lo, hi = 0, n_slices
    return np.arange(lo, hi, dtype=int)


def _resize_nearest(x2d: np.ndarray, out_hw: int):
    """Simple stride resize; fast and stable enough for features."""
    H, W = x2d.shape
    sh = max(1, H // out_hw)
    sw = max(1, W // out_hw)
    y = x2d[::sh, ::sw][:out_hw, :out_hw]
    if y.shape != (out_hw, out_hw):
        out = np.zeros((out_hw, out_hw), dtype=y.dtype)
        out[:y.shape[0], :y.shape[1]] = y
        y = out
    return y


def _ensure_k_unique(idx: np.ndarray, candidates: np.ndarray, k: int, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    idx = np.unique(idx.astype(int))
    candidates = np.unique(candidates.astype(int))
    idx = idx[np.isin(idx, candidates)]

    if len(candidates) <= k:
        return np.sort(candidates)

    if len(idx) >= k:
        keep = np.linspace(0, len(idx) - 1, k).round().astype(int)
        return np.sort(idx[keep])

    need = k - len(idx)
    remaining = candidates[~np.isin(candidates, idx)]
    if len(remaining) > 0:
        take = min(need, len(remaining))
        fill = rng.choice(remaining, size=take, replace=False)
        idx = np.unique(np.concatenate([idx, fill]))

    if len(idx) < k:
        need = k - len(idx)
        base = np.linspace(0, len(candidates) - 1, need).round().astype(int)
        idx = np.unique(np.concatenate([idx, candidates[base]]))

    if len(idx) > k:
        keep = np.linspace(0, len(idx) - 1, k).round().astype(int)
        idx = idx[keep]

    return np.sort(idx.astype(int))


def _slice_qc_metrics(slice_hu: np.ndarray):
    """
    Cheap QC + informativeness metrics from HU slice.
    Returns:
      lung_frac, air_frac, hi_dens_frac_in_lung, body_frac, mean_hu, std_hu
    """
    # Canonicalize HU first to remove domain differences across scans
    x = np.clip(slice_hu, HU_CANON_MIN, HU_CANON_MAX).astype(np.float32)

    lung_mask = (x < LUNG_HU_THR)
    lung_frac = float(lung_mask.mean())

    air_frac = float((x < -950).mean())

    if lung_mask.any():
        hi_dens_frac = float((x[lung_mask] > HI_DENS_THR).mean())
    else:
        hi_dens_frac = 0.0

    # NEW: body dominance proxy (higher => more soft tissue / less lung/background)
    body_frac = float((x > -300).mean())

    mean_hu = float(np.mean(x))
    std_hu  = float(np.std(x))
    return lung_frac, air_frac, hi_dens_frac, body_frac, mean_hu, std_hu


def sample_slice_indices_representative(
    vol: np.ndarray,
    n_select: int = N_SLICES_REP,
    center_fraction: float = CENTER_FRACTION,
    rng=None,
    feat_hw: int = FEAT_HW,
    n_pca: int = N_PCA,
    n_informative: int = N_INFORMATIVE,
):
    """
    Stronger representative sampling:
      1) central candidates
      2) remove junk slices using lung_frac/air_frac/body_frac
      3) always include top informative slices (hi_dens_frac in lung)
      4) fill remaining with diversity clustering (PCA + MiniBatchKMeans)
    """
    rng = np.random.default_rng(RANDOM_STATE) if rng is None else rng
    Z = int(vol.shape[0])
    candidates = _central_candidates(Z, center_fraction)

    if len(candidates) == 0:
        return np.array([max(0, Z // 2)], dtype=int)

    if len(candidates) <= n_select:
        return candidates.astype(int)

    # Stage A: QC filter + informativeness metrics
    metrics = []
    kept = []
    for z in candidates:
        s = vol[int(z)]
        lung_frac, air_frac, hi_dens_frac, body_frac, mean_hu, std_hu = _slice_qc_metrics(s)

        # drop junk extremes
        if lung_frac < MIN_LUNG_FRAC:
            continue
        if air_frac > MAX_AIR_FRAC:
            continue

        # NEW: drop body-dominant slices (reduces false positives like patient 2226)
        if body_frac > MAX_BODY_FRAC:
            continue

        kept.append(int(z))
        metrics.append((lung_frac, air_frac, hi_dens_frac, body_frac, mean_hu, std_hu))

    if len(kept) == 0:
        # fallback: no QC-passed slices -> go back to central candidates evenly
        base = np.linspace(0, len(candidates) - 1, n_select).round().astype(int)
        return np.sort(candidates[base]).astype(int)

    kept = np.asarray(kept, dtype=int)
    metrics = np.asarray(metrics, dtype=np.float32)

    # if after QC we have fewer than n_select, take all
    if len(kept) <= n_select:
        return np.sort(kept).astype(int)

    # Stage B1: include “informative” extremes
    # Use hi_dens_frac_in_lung as a crude proxy for consolidation / mucus / denser patterns
    hi_dens = metrics[:, 2]
    n_inf = int(min(n_informative, len(kept), max(8, n_select // 6)))
    informative_idx = np.argsort(-hi_dens)[:n_inf]  # top hi_dens
    chosen = kept[informative_idx]

    # Stage B2: diversity clustering on low-res normalized slice image
    # Use low-res pixels as diversity signal, but normalize first.
    xs = vol[kept].astype(np.float32)  # [M,H,W]
    xs = np.clip(xs, HU_CANON_MIN, 400.0)  # keep consistent, avoid weird floors
    xs = (xs - HU_CANON_MIN) / (400.0 - HU_CANON_MIN + 1e-6)  # -> [0,1]

    feats = np.stack([_resize_nearest(xi, feat_hw).reshape(-1) for xi in xs], axis=0)  # [M,F]
    feats = feats - feats.mean(axis=0, keepdims=True)
    feats = feats / (feats.std(axis=0, keepdims=True) + 1e-6)

    # PCA -> smaller, easier clustering
    try:
        from sklearn.decomposition import PCA
        pca = PCA(n_components=min(n_pca, feats.shape[1], feats.shape[0]-1), random_state=int(rng.integers(1e9)))
        feats_small = pca.fit_transform(feats)
    except Exception:
        feats_small = feats  # fallback

    # How many to pick via clustering after reserving informative
    remaining_k = int(max(0, n_select - len(np.unique(chosen))))
    if remaining_k <= 0:
        return _ensure_k_unique(chosen, kept, n_select, rng=rng)

    try:
        from sklearn.cluster import MiniBatchKMeans
        from sklearn.metrics import pairwise_distances_argmin_min

        km = MiniBatchKMeans(
            n_clusters=remaining_k,
            random_state=int(rng.integers(1e9)),
            batch_size=1024,
            n_init="auto",
            max_iter=200,
        )
        km.fit(feats_small)
        nearest, _ = pairwise_distances_argmin_min(km.cluster_centers_, feats_small)
        diverse = kept[nearest]

        idx = np.unique(np.concatenate([chosen, diverse]))
        idx = _ensure_k_unique(idx, candidates=kept, k=n_select, rng=rng)
        return idx.astype(int)

    except Exception as e:
        print("[warn] rep sampling clustering failed:", str(e))
        # fallback: fill evenly from QC-kept
        need = n_select - len(np.unique(chosen))
        base = np.linspace(0, len(kept) - 1, need).round().astype(int)
        idx = np.unique(np.concatenate([chosen, kept[base]]))
        idx = _ensure_k_unique(idx, candidates=kept, k=n_select, rng=rng)
        return idx.astype(int)



In [ ]:
# Cell 08B - Visualization of representative selection per patient

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IDX_PATH = f"{OUT_ROOT}/train_index.csv"
dfi = pd.read_csv(IDX_PATH)
dfi["patient_id"] = dfi["patient_id"].astype(str).str.zfill(4)

WINDOWS = {
    "lung": (-1350, 150),
    "mediastinum": (-160, 240),
}

def plot_rep_selection_for_patient(pid, n_show=18, seed=42):
    pid = str(pid).zfill(4)
    row = dfi[dfi["patient_id"] == pid].iloc[0]
    vp = row["volume_path"]
    vol = np.load(vp, mmap_mode="r")
    Z = vol.shape[0]

    rng = np.random.default_rng(seed)
    idx = sample_slice_indices_representative(
        vol, n_select=N_SLICES_REP, center_fraction=CENTER_FRACTION, rng=rng
    )
    idx = np.array(sorted(set(map(int, idx))))

    print(f"Patient {pid} | label={row['label']} | Z={Z} | selected={len(idx)}")
    print("z_min/z_max:", int(idx.min()), int(idx.max()))

    # coverage plot
    plt.figure(figsize=(10, 1.6))
    plt.scatter(idx, np.zeros_like(idx), s=15)
    plt.yticks([])
    plt.xlabel("z index")
    plt.title(f"{pid} representative z coverage")
    plt.show()

    # show slice grid (choose evenly-spaced subset from selected)
    if len(idx) > n_show:
        keep = np.linspace(0, len(idx)-1, n_show).round().astype(int)
        zs = idx[keep]
    else:
        zs = idx

    n = len(zs)
    fig, axes = plt.subplots(len(WINDOWS), n, figsize=(2.2*n, 2.2*len(WINDOWS)))
    if n == 1:
        axes = axes.reshape(len(WINDOWS), 1)

    for r_i, (wname, (vmin, vmax)) in enumerate(WINDOWS.items()):
        for c_i, z in enumerate(zs):
            ax = axes[r_i, c_i]
            ax.imshow(vol[int(z)], cmap="gray", vmin=vmin, vmax=vmax)
            ax.set_title(f"{wname}\nz={int(z)}")
            ax.axis("off")

    plt.tight_layout()
    plt.show()


plot_rep_selection_for_patient(\"REDACTED_PATIENT_ID\", n_show=20, seed=42)


In [ ]:
# Sampling of images for a given fold
def rep_stats_for_fold(fold=2, split="val", seed=42):
    df = dfi.copy()
    if split == "train":
        df = df[df["fold"] != fold]
    else:
        df = df[df["fold"] == fold]

    rows = []
    for _, r in df.iterrows():
        pid = r["patient_id"]
        vol = np.load(r["volume_path"], mmap_mode="r")
        rng = np.random.default_rng(seed + int(pid))
        idx = sample_slice_indices_representative(vol, n_select=N_SLICES_REP, center_fraction=CENTER_FRACTION, rng=rng)
        idx = np.array(sorted(set(map(int, idx))))
        rows.append({
            "patient_id": pid,
            "label": float(r["label"]),
            "Z_original": int(vol.shape[0]),
            "n_rep_selected": int(len(idx)),
            "z_min": int(idx.min()),
            "z_max": int(idx.max()),
            "rep_ratio": float(len(idx) / max(1, vol.shape[0])),
        })
    out = pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)
    return out

df_rep = rep_stats_for_fold(fold=4, split="val", seed=42)
display(df_rep)
print("min/median/max n_rep_selected:", df_rep["n_rep_selected"].min(), df_rep["n_rep_selected"].median(), df_rep["n_rep_selected"].max())


In [ ]:
# Cell 09 — Slice-level Dataset (representative sampling) for weakly-supervised training


!pip -q install torch torchvision

import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np


class SliceDatasetRepresentative(Dataset):
    def __init__(
        self,
        index_csv: str,
        fold: int,
        split: str,
        seed: int = 42,
        center_fraction: float=CENTER_FRACTION,
        n_select: int = N_SLICES_REP,
        return_z_norm: bool = False,
    ):
        """
        split: 'train' or 'val'
        fold: which fold is validation
        Returns one slice per __getitem__:
          x: (C,H,W)   (C=1 or 2 depending on preprocess_slice)
          y: float     patient label
          pid: str
          z: int       slice index in original volume
          z_norm: float in [0,1] position within volume (optional feature)
        """
        assert split in ["train", "val"]
        df = pd.read_csv(index_csv)
        df["patient_id"] = df["patient_id"].astype(str).str.zfill(4)
        df["label"] = df["label"].astype(float)
        df["fold"] = df["fold"].astype(int)

        if split == "train":
            self.df = df[df["fold"] != fold].reset_index(drop=True)
        else:
            self.df = df[df["fold"] == fold].reset_index(drop=True)

        self.split = split
        self.fold = int(fold)
        self.seed = int(seed)

        self.center_fraction = float(center_fraction)
        self.n_select = int(n_select)
        self.return_z_norm = bool(return_z_norm)

        # Build a flat "slice index" table:
        # one row per selected slice across all patients
        self.rep_cache = {}   # pid -> np.array of selected z indices
        self.rows = []        # list of (pid, label, volume_path, z, Z)

        for _, r in self.df.iterrows():
            pid = str(r["patient_id"]).zfill(4)
            y = float(r["label"])
            vp = str(r["volume_path"])

            # load just to get Z + rep indices (mmap is fine)
            vol = np.load(vp, mmap_mode="r")
            Z = int(vol.shape[0])

            rng = self._make_deterministic_rng(pid, "rep_train" if split == "train" else "rep_val")
            idx = sample_slice_indices_representative(vol,
                                                      rng=rng,
                                                      center_fraction=self.center_fraction,
                                                      n_select=self.n_select)

            if idx is None or len(idx) == 0:
                idx = np.array([max(0, Z // 2)], dtype=int)

            idx = np.asarray(idx, dtype=int)
            idx = idx[(idx >= 0) & (idx < Z)]
            idx = np.unique(idx)

            self.rep_cache[pid] = idx

            for z in idx:
                self.rows.append((pid, y, vp, int(z), Z))

        if len(self.rows) == 0:
            raise RuntimeError("No slices found. Check representative sampling and volume paths.")

        # training: shuffle slice order deterministically to avoid patient-block ordering
        if self.split == "train":
            rng = np.random.default_rng(self.seed + 1337 + self.fold)
            rng.shuffle(self.rows)

    def _make_deterministic_rng(self, pid: str, tag: str):
        s = (hash(f"{pid}|fold{self.fold}|{tag}|seed{self.seed}") & 0xffffffff)
        return np.random.default_rng(s)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        pid, y, vp, z, Z = self.rows[i]

        vol = np.load(vp, mmap_mode="r")          # (Z,H,W) HU
        sl = vol[int(z)]                          # (H,W)

        x = preprocess_slice(sl)                  # (C,H,W) OR (H,W) depending on cell 07
        x = np.asarray(x)

        # ensure channel-first tensor
        if x.ndim == 2:
            x = x[None, :, :]                     # (1,H,W)
        # if x.ndim == 3 it is already (C,H,W)

        x = torch.from_numpy(x).float()
        y = torch.tensor(float(y)).float()

        z_norm = 0.0 if Z <= 1 else float(z) / float(Z - 1)
        z_norm = torch.tensor(z_norm).float()

        if self.return_z_norm:
          return x, y, pid, int(z), z_norm
        return x, y, pid, int(z)


# quick sanity check
ds = SliceDatasetRepresentative(index_csv=f"{OUT_ROOT}/train_index.csv", fold=0, split="train", seed=42)
x, y, pid, z = ds[0]
print("Train slice example:", pid, "z=", z, "x:", tuple(x.shape), "y:", float(y))

ds_val = SliceDatasetRepresentative(index_csv=f"{OUT_ROOT}/train_index.csv", fold=0, split="val", seed=42)
x2, y2, pid2, z2 = ds_val[0]
print("Val slice example:", pid2, "z=", z2, "x:", tuple(x2.shape), "y:", float(y2))

print("Total train slices:", len(ds), " | Total val slices:", len(ds_val))



In [ ]:
# Cell 10A) Download RadImageNet PyTorch pretrained weights

!pip -q install gdown

import os, zipfile
import gdown
from pathlib import Path

OUT_ROOT = str(DATA_ROOT)
Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)

RADIMAGENET_URL = "https://drive.google.com/uc?id=1RHt2GnuOYlc_gcoTETtBDSW73mFyRAtR"
zip_path = f"{OUT_ROOT}/radimagenet_pytorch_models.zip"
dst_dir  = f"{OUT_ROOT}/radimagenet_weights"

if not os.path.exists(zip_path):
    gdown.download(RADIMAGENET_URL, zip_path, quiet=False)

Path(dst_dir).mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(dst_dir)


from glob import glob
files = sorted(glob(dst_dir + "/**/*", recursive=True))
print("Extracted files (first 50):")
for f in files[:50]:
    print(" -", f)
print("\nTotal extracted:", len(files))


In [ ]:
# Cell 10B — Slice model (2-channel input) with per-slice prediction

import torch
import torch.nn as nn
import torchvision.models as models


def _clean_state_dict(sd):
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    out = {}
    for k, v in sd.items():
        if k.startswith("module."):
            k = k[len("module."):]
        out[k] = v
    return out


def _strip_prefix(sd, prefix):
    out = {}
    for k, v in sd.items():
        out[k[len(prefix):]] = v if k.startswith(prefix) else v
        if not k.startswith(prefix):
            out[k] = v
    return out



class SliceRegressor(nn.Module):
    def __init__(self, backbone="resnet18", imagenet_pretrained=True, rad_weight_path=None):
        super().__init__()

        # input is 2-channel (lung + mediastinum); pretrained nets expect 3
        self.in_proj = nn.Conv2d(2, 3, kernel_size=1, bias=False)

        if backbone == "resnet18":
            net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if imagenet_pretrained else None)
            feat_dim = net.fc.in_features
            net.fc = nn.Identity()
            self.backbone = net

        elif backbone == "resnet50":
            net = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if imagenet_pretrained else None)
            self.backbone = nn.Sequential(
                net.conv1, net.bn1, net.relu, net.maxpool,
                net.layer1, net.layer2, net.layer3, net.layer4,
                net.avgpool, nn.Flatten(1)
            )
            feat_dim = 2048

        elif backbone == "densenet121":
            net = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT if imagenet_pretrained else None)
            feat_dim = net.classifier.in_features
            net.classifier = nn.Identity()
            self.backbone = net

        elif backbone == "inceptionv3":
            net = models.inception_v3(
                weights=models.Inception_V3_Weights.DEFAULT if imagenet_pretrained else None,
                aux_logits=False
            )
            feat_dim = net.fc.in_features
            net.fc = nn.Identity()
            self.backbone = net

        else:
            raise ValueError(f"Unsupported backbone: {backbone}")

        self.feat_norm = nn.LayerNorm(feat_dim)

        self.head = nn.Sequential(
            nn.Linear(feat_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

        if rad_weight_path is not None:
            ckpt = torch.load(rad_weight_path, map_location="cpu")
            sd = _clean_state_dict(ckpt)
            sd = _strip_prefix(sd, "backbone.")
            sd = {k: v for k, v in sd.items() if not k.startswith("AuxLogits.")}

            if backbone == "densenet121" and any(k.startswith("0.") for k in sd.keys()):
                sd = {("features." + k[2:]) if k.startswith("0.") else k: v for k, v in sd.items()}

            missing, unexpected = self.backbone.load_state_dict(sd, strict=False)
            print(f"[RadImageNet] loaded weights from: {rad_weight_path}")
            print("  missing keys (first 10):", list(missing)[:10])
            print("  unexpected keys (first 10):", list(unexpected)[:10])


    def forward(self, x):
        """
        Slice-level forward
        x: [B,2,H,W]  (a batch of slices)
        returns: [B] predictions
        """
        x = self.in_proj(x)
        f = self.backbone(x)
        f = torch.nan_to_num(f, nan=0.0, posinf=0.0, neginf=0.0)
        f = self.feat_norm(f)
        s = self.head(f).squeeze(-1)  # [B]
        return s


    forward_slice = forward




In [ ]:
# Cell 11 — Patient-level evaluation from slice loader

import numpy as np
import torch
from collections import defaultdict
from scipy.stats import spearmanr, pearsonr



def aggregate_scores(scores, mode="mean", topk_frac=0.2, topk_k=None, mix=0.7):
    scores = np.asarray(scores, dtype=float)
    if len(scores) == 0:
        return 0.0

    if mode == "mean":
        return float(scores.mean())

    if topk_k is not None:
        k = int(topk_k)
    else:
        k = int(max(1, int(len(scores) * float(topk_frac))))
    k = max(1, min(k, len(scores)))

    top = float(np.sort(scores)[-k:].mean())
    if mode == "topk":
        return top
    if mode == "mix":
        return float(mix * top + (1.0 - mix) * scores.mean())

    raise ValueError("mode must be mean/topk/mix")



def eval_patient_level_from_slices(
    model,
    loader,
    device,
    agg_mode="topk",
    topk_frac=0.2,
    topk_k=None,
    mix=0.7
):
    model.eval()
    preds_by_pid = defaultdict(list)
    y_by_pid = {}

    with torch.no_grad():
        for batch in loader:
            # batch may be (x,y,pid,z) or (x,y,pid,z,z_norm)
            x, y, pid, z = batch[0], batch[1], batch[2], batch[3]

            x = x.to(device)  # [B,2,H,W]
            preds = model(x).detach().cpu().numpy().astype(float)  # [B]

            # y can be tensor [B]
            y_np = y.detach().cpu().numpy().astype(float)

            # pid is list[str] length B
            for i in range(len(preds)):
                p = pid[i]
                preds_by_pid[p].append(float(preds[i]))
                y_by_pid[p] = float(y_np[i])  # same label for all slices of that patient

    pids = sorted(preds_by_pid.keys())
    y_true = np.array([y_by_pid[p] for p in pids], dtype=float)
    y_pred = np.array(
        [
            aggregate_scores(
                preds_by_pid[p],
                mode=agg_mode,
                topk_frac=topk_frac,
                topk_k=topk_k,
                mix=mix,
            )
            for p in pids
        ],
        dtype=float,
    )

    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

    sse = float(np.sum((y_true - y_pred) ** 2))
    sst = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1.0 - sse / (sst + 1e-12))

    if np.std(y_true) < 1e-12 or np.std(y_pred) < 1e-12:
        rho, r = 0.0, 0.0
    else:
        rho = spearmanr(y_true, y_pred).correlation
        rho = 0.0 if (rho is None or np.isnan(rho)) else float(rho)
        r = pearsonr(y_true, y_pred)[0]
        r = 0.0 if (r is None or np.isnan(r)) else float(r)

    return {"MAE": mae, "RMSE": rmse, "R2": r2, "Spearman": rho, "Pearson": r}




In [ ]:
# Cell 12 — Slice-level training + patient-level validation early stopping

import torch
import numpy as np
from torch.utils.data import DataLoader
import torchvision.transforms.functional as TF


def augment_ct_batch(x, max_deg=5, max_shift=6, jitter=0.03, noise=0.02):
    """
    x: [B,2,H,W] in [0,1]
    Applies per-sample augmentation.
    """
    B = x.shape[0]
    out = []
    for b in range(B):
        xb = x[b]  # [2,H,W]
        angle = float(np.random.uniform(-max_deg, max_deg))
        tx = int(np.random.uniform(-max_shift, max_shift))
        ty = int(np.random.uniform(-max_shift, max_shift))
        scale = float(np.random.uniform(0.98, 1.02))

        # apply same affine to both channels
        chs = []
        for c in range(xb.shape[0]):
            img = xb[c:c+1]  # [1,H,W]
            img = TF.affine(img, angle=angle, translate=[tx, ty], scale=scale, shear=[0.0, 0.0])
            chs.append(img)
        xb = torch.cat(chs, dim=0)  # [2,H,W]

        xb = torch.clamp(
            xb * float(np.random.uniform(1.0 - jitter, 1.0 + jitter)) + float(np.random.uniform(-jitter, jitter)),
            0.0, 1.0
        )
        xb = torch.clamp(xb + noise * torch.randn_like(xb), 0.0, 1.0)
        out.append(xb)
    return torch.stack(out, dim=0)


def set_backbone_trainable(model, trainable: bool):
    for p in model.backbone.parameters():
        p.requires_grad = trainable


def set_partial_trainable(model, trainable: bool):
    bb = model.backbone
    if isinstance(bb, torch.nn.Sequential) and len(bb) >= 8:
        for p in bb[7].parameters():
            p.requires_grad = trainable
        return
    if hasattr(bb, "layer4"):
        for p in bb.layer4.parameters():
            p.requires_grad = trainable
        return
    if hasattr(bb, "features") and hasattr(bb.features, "denseblock4"):
        for p in bb.features.denseblock4.parameters():
            p.requires_grad = trainable
        if hasattr(bb.features, "norm5"):
            for p in bb.features.norm5.parameters():
                p.requires_grad = trainable
        return
    print("[warn] unknown backbone structure for partial unfreeze; unfreezing whole backbone.")
    set_backbone_trainable(model, trainable)


def train_fold_es_slicelevel(
    fold=0,
    max_epochs=20,
    patience=5,
    lr=1e-4,
    batch_size=32,
    freeze_epochs=3,
    dropout=0.3,
    seed=42,
    grad_clip=1.0,
    wd=2e-4,
    use_aug=True,
    # patient-level aggregation for validation
    agg_mode="topk",
    topk_frac=0.2,
    topk_k=20,
    mix=0.7,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    index_csv = f"{OUT_ROOT}/train_index.csv"
    train_ds = SliceDatasetRepresentative(index_csv=index_csv, fold=fold, split="train", seed=seed, center_fraction=CENTER_FRACTION)
    val_ds   = SliceDatasetRepresentative(index_csv=index_csv, fold=fold, split="val",   seed=seed, center_fraction=CENTER_FRACTION)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0)



    backbone = "resnet18"  # "resnet18" | "resnet50" | "densenet121" | "inceptionv3"

    RAD_PATHS = {
        "resnet50": "/path/to/authorised/data/radimagenet_weights/RadImageNet_pytorch/ResNet50.pt",
        "densenet121": "/path/to/authorised/data/radimagenet_weights/RadImageNet_pytorch/DenseNet121.pt",
        "inceptionv3": "/path/to/authorised/data/radimagenet_weights/RadImageNet_pytorch/InceptionV3.pt",
}

    rad_path = RAD_PATHS.get(backbone, None)  # will be None if backbone not in dict


    # NOTE: SliceRegressor in Cell 10C does NOT accept in_ch; it always assumes 2-channel input.
    model = SliceRegressor(
        backbone=backbone,
        imagenet_pretrained=(rad_path is None),
        rad_weight_path=rad_path,
    ).to(device)

    # set dropout
    for m in model.modules():
        if isinstance(m, torch.nn.Dropout):
            m.p = dropout

    # freeze backbone initially
    set_backbone_trainable(model, False)
    for p in model.head.parameters():
        p.requires_grad = True
    for p in model.in_proj.parameters():
        p.requires_grad = True

    loss_fn = torch.nn.MSELoss()
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=wd)

    best = {"epoch": -1, "RMSE": np.inf, "state": None, "metrics": None}
    bad_epochs = 0

    for ep in range(1, max_epochs + 1):

        if ep == freeze_epochs + 1:
            set_partial_trainable(model, True)
            # small LR drop after unfreeze is OK; feel free to remove if you want faster finetune
            opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr * 0.3, weight_decay=wd)

        model.train()
        losses = []

        for batch in train_loader:
            # batch may be (x,y,pid,z) or (x,y,pid,z,z_norm)
            x, y, pid, z = batch[0], batch[1], batch[2], batch[3]

            x = x.to(device)  # [B,2,H,W]
            y = y.to(device)  # [B]

            if use_aug:
                x = augment_ct_batch(x)

            pred = model(x)  # [B]
            loss = loss_fn(pred, y)

            opt.zero_grad()
            loss.backward()
            if grad_clip is not None and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()

            losses.append(float(loss.item()))

        # patient-level validation
        val_metrics = eval_patient_level_from_slices(
            model, val_loader, device,
            agg_mode=agg_mode, topk_frac=topk_frac, topk_k=topk_k, mix=mix
        )

        print(
            f"Fold {fold} | Epoch {ep:02d} | train MSE {np.mean(losses):.4f} | "
            f"VAL(pat) MAE {val_metrics['MAE']:.3f} | RMSE {val_metrics['RMSE']:.3f} | "
            f"R2 {val_metrics['R2']:.3f} | Spearman {val_metrics['Spearman']:.3f}"
        )

        if val_metrics["RMSE"] + 1e-6 < best["RMSE"]:
            best = {
                "epoch": ep,
                "RMSE": val_metrics["RMSE"],
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "metrics": val_metrics,
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print(
                f"Early stop: no RMSE improvement for {patience} epochs. "
                f"Best epoch={best['epoch']} RMSE={best['RMSE']:.3f}"
            )
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"], strict=True)

    return best["metrics"], best["epoch"], model




In [ ]:
# Cell 13 — 5-fold CV (slice-level training, patient-level evaluation)

import pandas as pd

results = []
best_epochs = []

for fold in range(5):
    metrics, best_ep = train_fold_es_slicelevel(
        fold=fold,
        max_epochs=20,
        patience=5,
        lr=1e-4,
        batch_size=32,
        freeze_epochs=3,
        dropout=0.3,
        seed=42,
        wd=2e-4,
        use_aug=True,
        agg_mode="mix",
        mix=0.7,
        topk_k=20,
        topk_frac=0.15,
    )
    results.append({"fold": fold, **metrics})
    best_epochs.append(best_ep)
    print("Fold", fold, "best epoch:", best_ep, "metrics:", metrics)

df_res = pd.DataFrame(results)
print("\nPer-fold results:")
display(df_res)

print("\nSummary (mean ± std):")
for col in ["MAE","RMSE","R2","Spearman","Pearson"]:
    m = df_res[col].mean()
    s = df_res[col].std(ddof=1)
    print(f"{col}: {m:.3f} ± {s:.3f}")

print("\nBest epochs per fold:", best_epochs)

